# Notebook 2: Sponsorship Classification

**Business problem:** How do we know which videos are sponsored?
YouTube's paid-promotion label is not detectable via public API.
This notebook builds and validates a regex-based heuristic classifier
to label every video as Organic, Implicit, or Explicit sponsorship.

**Validation:** Compared against 100 manually labelled videos.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

DATA_DIR = '../data/'
df = pd.read_csv(DATA_DIR + 'video_engagement_rate_data.csv')
print(df.shape)


In [ ]:
Sample_df = pd.read_csv('../data/Sample Validation.csv')

# Update manual_sponsored based on your manual label
Sample_df['hasSponsorKeywords'] = Sample_df['Sponsorship'].map(lambda x: True if x in ['Explicit', 'Implicit'] else False)
Sample_df


In [ ]:
# 1. Distribution by class (manual label)
class_counts = Sample_df['Sponsorship'].value_counts()

# 2. Sample representative examples for each class
examples = {}
for label in ['Explicit', 'Implicit', 'Organic']:
    subset = Sample_df[Sample_df['Sponsorship'] == label].head(5)  # Take first 3 as sample
    examples[label] = subset[['title', 'description', 'publishedAt']].to_dict(orient='records')

class_counts, examples

In [ ]:
#Sponsorship Classification

# 1. Regex Patterns
explicit_pattern = re.compile(r"""
    (thanks?\s*(to)?\s*\w*\s*for sponsoring)|
    (thank you.*(sponsor|sponsoring|sponsored by|for sponsoring))|
    ((this video|segment|channel|content) is sponsored by)|
    (sponsor(ed|ing)?( by)?[:\s-])|
    (ad[.:\s-])|
    (paid promotion)|
    (presented by)|
    (in partnership with)|
    (promo ?code)|
    (ad segment)|
    (ad break)|
    (\#ad)|
    (\#sponsored)|
    (advertisement)
""", re.IGNORECASE | re.VERBOSE)

# "implicit"(only matches affiliate/shortened URLs etc.)
implicit_pattern = re.compile(r"""
    ((provided|supplied|sent|gifted) by [A-Za-z0-9&\-\s]+( for (review|testing|this video))?)|
    ((get|buy|shop|check out|purchase|order|grab)[^\n]{0,40}(https?://(amzn\.to|geni\.us|bit\.ly|rstyle\.me|shopmy\.us|kckb\.me|amazon\.com)\S*))|
    (affiliate (link|program|disclosure|code|partnership))|
    ((referral link|discount code|save \d+%|use code|with code|commission|small commission))|
    (amazon\.com|amzn\.to|geni\.us|bit\.ly|rstyle\.me|shopmy\.us|kckb\.me)|
    (merch[:\s\-]*(https?://(amzn\.to|geni\.us|bit\.ly|rstyle\.me|shopmy\.us|kckb\.me|amazon\.com)\S*))|
    (store[:\s\-]*(https?://(amzn\.to|geni\.us|bit\.ly|rstyle\.me|shopmy\.us|kckb\.me|amazon\.com)\S*))|
    (shop[:\s\-]*(https?://(amzn\.to|geni\.us|bit\.ly|rstyle\.me|shopmy\.us|kckb\.me|amazon\.com)\S*))
""", re.IGNORECASE | re.VERBOSE)

# 2. Debuggable Classifier
def classify_sponsorship_debug(text):
    match_exp = explicit_pattern.search(text)
    if match_exp:
        return "Explicit", f"Explicit: {match_exp.group(0)}"
    match_imp = implicit_pattern.search(text)
    if match_imp:
        return "Implicit", f"Implicit: {match_imp.group(0)}"
    return "Organic", "Organic: no match"

def get_full_text(row):
    title = str(row['title']) if 'title' in row else ''
    description = str(row['description']) if 'description' in row else ''
    tags = str(row['tags']) if 'tags' in row else ''
    return ' '.join([title, description, tags])

# 3. Fill missing
for col in ['title','description','tags']:
    if col in traditional_videos_df.columns:
        traditional_videos_df[col] = traditional_videos_df[col].fillna('').astype(str)

# 4. Run classifier & debug info
traditional_videos_df[['Sponsorship','_match_info']] = traditional_videos_df.apply(lambda row: pd.Series(classify_sponsorship_debug(get_full_text(row))),axis=1)
traditional_videos_df['hasSponsorKeywords'] = traditional_videos_df['Sponsorship'].map(lambda x: x in ['Explicit', 'Implicit'])

# 5. Merge with sample manual labels
comparison = traditional_videos_df.merge(
    Sample_df[['videoId', 'hasSponsorKeywords']],
    on='videoId',
    how='inner',
    suffixes=('','_manual')
)
comparison['hasSponsorKeywords'] = comparison['hasSponsorKeywords'].astype(int)
comparison['hasSponsorKeywords_manual'] = comparison['hasSponsorKeywords_manual'].astype(int)

# 6. Evaluation
print("Value Counts (Auto):")
print(comparison['hasSponsorKeywords'].value_counts())
print("Value Counts (Manual):")
print(comparison['hasSponsorKeywords_manual'].value_counts())
print("\nConfusion Matrix:")
print(confusion_matrix(
    comparison['hasSponsorKeywords_manual'],
    comparison['hasSponsorKeywords'],
    labels=[1, 0]
))
print("\nClassification Report:")
print(classification_report(comparison['hasSponsorKeywords_manual'],comparison['hasSponsorKeywords']))

# 7. Debug Misclassifications
print("\nFalse Negatives (Missed Sponsors):")
print(comparison[(comparison['hasSponsorKeywords_manual']==1) & (comparison['hasSponsorKeywords']==0)][['videoId','title','description']])
print("\nFalse Positives (Organic marked as Sponsored):")
print(comparison[(comparison['hasSponsorKeywords_manual']==0) & (comparison['hasSponsorKeywords']==1)][['videoId','title','description']])


# 9. Show some organic examples
organic = traditional_videos_df[traditional_videos_df['Sponsorship'] == 'Organic']
print("\nOrganic Examples:")
print(organic[['videoId', 'channelId', 'title','description','publishedAt', 'Sponsorship']].head(10))


In [ ]:
mismatches = comparison[comparison['hasSponsorKeywords'] != comparison['hasSponsorKeywords_manual']]
organic_misclassified = mismatches[mismatches['hasSponsorKeywords'] == 'False']


for idx, row in mismatches.iterrows():
    if row['hasSponsorKeywords'] == True and row['hasSponsorKeywords_manual'] == False:
        text = get_full_text(row)
        match = explicit_pattern.search(text)
        print("\n-----")
        print(f"videoId: {row['videoId']}")
        print(f"Title: {row['title']}")
        print(f"Matched: {match.group() if match else None}")
        print(f"Text: {text[:250]}...")  # Print first 250 chars for context


In [ ]:
for label in ['Explicit', 'Implicit', 'Organic']:
    sample = traditional_videos_df[traditional_videos_df['Sponsorship']==label].sample(10, random_state=42)
    sample


In [ ]:
df = traditional_videos_df.copy()
df = df.sort_values(['creatorName', 'publishedAt']).reset_index(drop=True)

In [ ]:
# 1. First sponsorship occurrence per creator (using hasSponsorKeywords)
# Calculate first sponsorship date for each creator
first_sponsor = df[df['hasSponsorKeywords']].groupby('creatorName')['publishedAt'].min().rename('first_sponsor_date')
df = df.merge(first_sponsor, on='creatorName', how='left')

# After merge, first_sponsor_date may be all NaN for some creators with NO sponsored videos
# To avoid KeyError, handle missing 'first_sponsor_date' in further calculations
df['is_pre_sponsor'] = df.apply(
    lambda row: pd.isnull(row['first_sponsor_date']) or row['publishedAt'] < row['first_sponsor_date'],
    axis=1
)

# When no sponsorship, videos_since_first_sponsor is always NaN
def get_videos_since_first_sponsor(row):
    # No sponsorship for this creator
    if pd.isnull(row['first_sponsor_date']):
        return np.nan
    # This video is before the first sponsorship
    if row['publishedAt'] < row['first_sponsor_date']:
        return np.nan
    # Count videos (for this creator) since first sponsorship up to this one
    creator_videos = df[df['creatorName'] == row['creatorName']]
    return (creator_videos['publishedAt'] <= row['publishedAt']).sum() - \
           (creator_videos['publishedAt'] < row['first_sponsor_date']).sum()
df['videos_since_first_sponsor'] = df.apply(get_videos_since_first_sponsor, axis=1)

# Days since first sponsorship (NaN if no sponsorship ever)
df['days_since_first_sponsor'] = (df['publishedAt'] - df['first_sponsor_date']).dt.days
df.loc[df['is_pre_sponsor'], 'days_since_first_sponsor'] = np.nan


# 2. Time splits and temporal features
df['upload_hour'] = df['publishedAt'].dt.hour
df['day_of_week'] = df['publishedAt'].dt.day_name()
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)
df['year_month'] = df['publishedAt'].dt.to_period('M')


# 3. Days since published (relative to a cutoff, e.g., today or scrape date)
REF_DATE = pd.Timestamp('2025-07-26', tz='UTC')
df['days_since_published'] = (REF_DATE - df['publishedAt']).dt.days.clip(lower=0)


# 4. Engagement per day (normalize for video age)
for col in ['viewCount', 'likeCount', 'commentCount']:
    df[f'{col}_per_day'] = df[col] / (df['days_since_published'].replace(0, 1))  # avoid div by zero


# 5. Rolling engagement stats per creator (window of 5 previous videos)
for col in ['viewCount', 'likeCount', 'commentCount']:
    df[f'{col}_rolling5_mean'] = df.groupby('creatorName')[col].transform(lambda x: x.shift().rolling(5, min_periods=1).mean())


# 6. Number of sponsored videos in past 5 uploads
df['sponsored_rolling5'] = df.groupby('creatorName')['hasSponsorKeywords'].transform(
    lambda x: x.shift().rolling(5, min_periods=1).sum())



# 7. Length of title and description
df['title_len'] = df['title'].str.len()
df['description_len'] = df['description'].str.len()


# 8. Engagement Metrics
# Avoid division by zero
df['like_engagement_rate'] = df['likeCount'] / df['viewCount'].replace(0, np.nan)
df['comment_engagement_rate'] = df['commentCount'] / df['viewCount'].replace(0, np.nan)
df['total_engagement_rate'] = (df['likeCount'] + df['commentCount']) / df['viewCount'].replace(0, np.nan)



# 9. Monthly sponsor/organic ratio per creator
monthly = df.groupby(['creatorName', 'year_month'])['hasSponsorKeywords'].agg(
    sponsored='sum', total='count').reset_index()
monthly['pct_sponsored'] = monthly['sponsored'] / monthly['total']
# Optionally, merge back to df:
df = df.merge(monthly[['creatorName', 'year_month', 'pct_sponsored']], on=['creatorName', 'year_month'], how='left')



# 10. Cumulative number of sponsored videos up to this video (per creator)
df['cum_sponsored'] = df.groupby('creatorName')['hasSponsorKeywords'].cumsum()



# 11. Proportion of sponsored/organic videos per creator (last 6 months or all)
creator_stats = df.groupby('creatorName').agg(
    n_sponsored=('hasSponsorKeywords', 'sum'),
    n_organic=('hasSponsorKeywords', lambda x: (~x).sum()),
    total_vids=('videoId', 'count')
)
creator_stats['pct_sponsored'] = creator_stats['n_sponsored'] / creator_stats['total_vids']
creator_stats['pct_organic'] = creator_stats['n_organic'] / creator_stats['total_vids']
df = df.merge(creator_stats[['pct_sponsored', 'pct_organic']], left_on='creatorName', right_index=True, how='left')



# 12. Days since first sponsorship (NaN if before sponsorship)
df['days_since_first_sponsor'] = (df['publishedAt'] - df['first_sponsor_date']).dt.days
df.loc[df['is_pre_sponsor'], 'days_since_first_sponsor'] = np.nan



# 13. Video duration in minutes (if duration_sec exists)
if 'duration_sec' in df.columns:
    df['duration_min'] = df['duration_sec'] / 60


# 14. Log-transform engagement variables (avoid issues with skewness)
for col in ['viewCount', 'likeCount', 'commentCount']:
    df[f'log_{col}'] = np.log1p(df[col])


df

In [ ]:
df.info()
df.describe()

In [ ]:
#Distribution of Sponsored vs Organic videos per creator by genre/size

# Count sponsored/organic per creator
counts = df.groupby(['creatorName', 'sizeCategory', 'genre', 'hasSponsorKeywords']).size().unstack(fill_value=0)
counts.columns = ['organic', 'sponsored'] if 'sponsored' in counts.columns else ['organic', 'sponsored']
counts = counts.reset_index()

# Bar plot: stacked for each creator
counts_sorted = counts.sort_values('sponsored', ascending=False)
plt.figure(figsize=(14,7))
plt.bar(counts_sorted['creatorName'], counts_sorted['organic'], label='Organic', color='#62c370')
plt.bar(counts_sorted['creatorName'], counts_sorted['sponsored'], bottom=counts_sorted['organic'], label='Sponsored', color='#3a7ca5')
plt.xticks(rotation=80, ha='right', fontsize=9)
plt.xlabel('Creator Name')
plt.ylabel('Number of Videos')
plt.title('Organic vs Sponsored Videos per Creator in last year')
plt.legend()
plt.tight_layout()
plt.show()

# Print out the table for reference
print("\nOrganic and Sponsored Video Counts by Creator in last year:")
print(counts_sorted[['creatorName', 'sizeCategory', 'genre', 'organic', 'sponsored']])

In [ ]:
# In case your df doesn't have a boolean-to-string mapping for sponsorship:
df['sponsor_label'] = df['hasSponsorKeywords'].map({True: 'Sponsored', False: 'Organic'})

# Total counts
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='sponsor_label', palette='Set2')
plt.title('Number of Videos: Sponsored vs Organic')
plt.xlabel('Type'); plt.ylabel('Count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
sns.histplot(data=df, x='total_engagement_rate', hue='Sponsorship', element='step', bins=40, kde=True, common_norm=False)
plt.xlim(0, 0.2)  # Engagement rates are typically small, adjust as needed
plt.xlabel('Engagement Rate')
plt.title('Engagement Rate Distribution by Sponsorship Type')
plt.show()


plt.figure(figsize=(8,4))
sns.boxplot(data=df, x='Sponsorship', y='total_engagement_rate', showfliers=False, palette='Set2')
plt.ylabel('Total Engagement Rate')
plt.title('Total Engagement Rate by Sponsorship Type')
plt.show()


# By Genre
plt.figure(figsize=(8,4))
sns.countplot(data=df, x='genre', hue='sponsor_label', palette='Set2')
plt.title('Sponsored vs Organic by Genre')
plt.xlabel('Genre'); plt.ylabel('Count')
plt.legend(title='Type')
plt.tight_layout()
plt.show()

# By Creator (Top 20 for readability)
creator_counts = df.groupby(['creatorName', 'sponsor_label']).size().unstack(fill_value=0)
creator_counts = creator_counts.sort_values('Sponsored', ascending=False)
creator_counts.iloc[:20].plot(kind='bar', stacked=True, figsize=(14,6))
plt.title('Organic vs Sponsored per Creator (Top 20)')
plt.ylabel('Number of Videos')
plt.tight_layout()
plt.show()


In [ ]:
# Monthly trend: Total and split by sponsorship
monthly = df.groupby(['year_month', 'sponsor_label']).size().unstack(fill_value=0)
monthly.plot(kind='bar', stacked=True, figsize=(16,5), colormap='Set2')
plt.title('Monthly Video Uploads by Sponsorship')
plt.xlabel('Month'); plt.ylabel('Number of Videos')
plt.tight_layout()
plt.show()

# Line plot of engagement over time (median)
monthly_engagement = df.groupby(['year_month', 'sponsor_label'])['viewCount'].median().unstack()
monthly_engagement.plot(figsize=(14,5))
plt.title('Median Views per Video (Monthly, by Type)')
plt.ylabel('Median Views')
plt.xlabel('Month')
plt.tight_layout()
plt.show()


In [ ]:
for col in ['viewCount', 'likeCount', 'commentCount']:
    plt.figure(figsize=(7,4))
    sns.boxplot(data=df, x='sponsor_label', y=col, showfliers=False, palette='Set2')
    plt.title(f'{col} Distribution by Sponsorship')
    plt.yscale('log')
    plt.xlabel('Type')
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

# Histograms (log scale)
for col in ['viewCount', 'likeCount', 'commentCount']:
    plt.figure(figsize=(8,4))
    for label, color in zip(['Sponsored','Organic'], ['orange','green']):
        subset = df[df['sponsor_label'] == label]
        plt.hist(np.log1p(subset[col]), bins=30, alpha=0.6, label=label, color=color)
    plt.title(f'Log-scaled Histogram of {col} by Type')
    plt.xlabel(f'log1p({col})')
    plt.ylabel('Number of Videos')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# For a few creators, plot rolling mean of views
sample_creators = df['creatorName'].value_counts().index[:3]  # Top 3 creators
for creator in sample_creators:
    sub = df[df['creatorName'] == creator].sort_values('publishedAt')
    plt.figure(figsize=(12,5))
    plt.plot(sub['publishedAt'], sub['viewCount_rolling5_mean'], label='Rolling Mean Views (5 videos)', color='purple')
    plt.scatter(sub['publishedAt'], sub['viewCount'], c=sub['hasSponsorKeywords'].map({True:'red',False:'green'}), alpha=0.5, label='Actual Views')
    plt.title(f'Views Over Time for {creator}')
    plt.xlabel('Date'); plt.ylabel('Views')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x='sponsor_label', y='title_len', palette='Set2')
plt.title('Title Length by Sponsorship')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=df, x='sponsor_label', y='description_len', palette='Set2')
plt.title('Description Length by Sponsorship')
plt.tight_layout()
plt.show()


In [ ]:
# Upload hour distribution by sponsorship
plt.figure(figsize=(10,4))
sns.histplot(data=df, x='upload_hour', hue='sponsor_label', multiple='stack', bins=24, palette='Set2')
plt.title('Upload Hour by Sponsorship')
plt.tight_layout()
plt.show()

# Day of week
plt.figure(figsize=(8,4))
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
sns.countplot(data=df, x='day_of_week', hue='sponsor_label', order=order, palette='Set2')
plt.title('Upload Day by Sponsorship')
plt.tight_layout()
plt.show()


In [ ]:
# pct_sponsored per creator
plt.figure(figsize=(12,4))
df_pct = df.groupby('creatorName')['hasSponsorKeywords'].mean().sort_values()
plt.bar(df_pct.index, df_pct.values, color='purple')
plt.title('Proportion of Sponsored Videos per Creator')
plt.ylabel('Pct Sponsored')
plt.xlabel('Creator Name')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
# Engagement and sponsorship timeline for a creator
example = df[df['creatorName'] == sample_creators[0]].sort_values('publishedAt')
plt.figure(figsize=(14,5))
plt.plot(example['publishedAt'], example['viewCount'], label='Views', color='blue', marker='o')
plt.scatter(example['publishedAt'], example['viewCount'], c=example['hasSponsorKeywords'].map({True:'red',False:'green'}), s=40)
plt.title(f'Views Timeline with Sponsorship Overlay ({sample_creators[0]})')
plt.xlabel('Date'); plt.ylabel('Views')
plt.tight_layout()
plt.show()


In [ ]:
medians = df.groupby(['year_month','sponsor_label'])['total_engagement_rate'].median().reset_index()
plt.figure(figsize=(14,5))
for label in medians['sponsor_label'].unique():
    tmp = medians[medians['sponsor_label']==label]
    plt.plot(tmp['year_month'].astype(str), tmp['total_engagement_rate'], label=label)
plt.legend()
plt.ylabel('Median Engagement Rate')
plt.xlabel('Month')
plt.title('Median Engagement Rate per Video by Month and Sponsorship')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
#Upload Hour Distribution: Overall
plt.figure(figsize=(10,4))
sns.histplot(df['upload_hour'], bins=24, kde=True, color='royalblue')
plt.title('Distribution of Video Upload Hours (Traditional Videos)')
plt.xlabel('Hour of Day (UTC)')
plt.ylabel('Number of Videos')
plt.show()

#Peak Upload Hour Table
# Find most common upload hour per creator
peak_hour = df.groupby('creatorName')['upload_hour'].agg(lambda x: x.value_counts().index[0])
print("Peak upload hour per creator:")
print(peak_hour)

In [ ]:
monthly_engage = df.groupby(['year_month','sponsor_label']).agg(
    mean_engage=('total_engagement_rate','mean'),
    median_engage=('total_engagement_rate','median'),
    video_count=('videoId','count')
).reset_index()

display(monthly_engage)


In [ ]:
df=df.drop(columns=['title', 'description', '_match_info', 'sizeCategory', 'duration_sec', 'pct_sponsored_x', 'sponsor_label'], axis=1)
df

In [ ]:
df.columns